In [1]:
import numpy as np
import scipy.stats


# ANCOVA
def std_F(F, npv):
    return 2 * np.sqrt(F) * npv


def std_p_F(F, df, npv):
    return 2 * np.sqrt(F) * scipy.stats.f.pdf(F, dfn=1, dfd=df) * npv


# T-test
def std_t(npv):
    return npv


def std_p_t(t, df, npv):
    return 2 * scipy.stats.t.pdf(np.abs(t), df) * npv


# Partial correlation
def var_r(r, n, npvr):
    return ((1 - r**2) / (n - 1)) * npvr**2


def std_r(r, n, npvr):
    var = var_r(r, n, npvr)
    return np.sqrt(var)


def t_from_r(r, n):
    return r * np.sqrt((n - 2) / (1 - r**2))


def std_p(r, n, npvr):
    t = t_from_r(r, n)
    return (
        2
        * scipy.stats.t.pdf(np.abs(t), df=n - 2)
        * np.sqrt((n - 2) / (n - 1))
        * npvr
        / (1 - r**2)
    )

In [10]:
y= std_p_F(6,75,0.1691)
y

np.float64(0.007219936122005526)

In [4]:
from scipy.stats import beta, f, t
def pvalue_ttest(t_stat, df):
    """Return the two-sided p-value for an already-computed t statistic."""
    return 2 * t.sf(np.abs(t_stat), df)


def pvalue_partial_corr(r, df):
    """Return a two-sided partial-correlation p-value from r and its t df."""
    r = np.asarray(r, dtype=float)
    if np.any(np.abs(r) >= 1):
        raise ValueError("Partial-correlation coefficients must be strictly between -1 and 1.")
    t_stat = r * np.sqrt(df / (1 - r**2))
    return 2 * t.sf(np.abs(t_stat), df)


def pvalue_ancova(F_stat, df2, df1=1):
    """Return the upper-tail p-value for an ANCOVA (F-test) statistic."""
    return f.sf(F_stat, df1, df2)


In [11]:
p=pvalue_ancova(6,75,1)
p

np.float64(0.016639791628428845)

In [12]:


def beta_params_from_mean_var(mu, var, eps=1e-12):
    """Return (a,b) for Beta by moment matching. Clamps var to < mu(1-mu)."""
    max_var = mu * (1 - mu) - eps
    # Clamp var for non-degenerate cases
    var_clamped = np.minimum(var, max_var)

    # Calculate k for non-degenerate cases
    # Ensure we don't divide by zero if var_clamped becomes zero due to clamping
    # This can happen if mu*(1-mu) is very small and var is also very small but positive.
    # The original code implicitly handles this by setting k=1e6 when var<=0.
    # Here, we ensure that if var_clamped is effectively zero, k is large.
    # A small positive value for the denominator to avoid division by zero for k calculation
    denominator = np.where(var_clamped == 0, eps, var_clamped)
    k_calculated = mu * (1 - mu) / denominator - 1.0
    # Ensure k is not negative, as it should be > 0 for Beta distribution
    k_calculated = np.maximum(k_calculated, eps)  # k must be > 0

    a_calculated = mu * k_calculated
    b_calculated = (1 - mu) * k_calculated

    return a_calculated, b_calculated


# def flip_proba_beta(p0, sigma_P, alpha=0.05):
#     """
#     Flip probability at alpha with P ~ Beta(a,b).
#     Provide either sigma_P.
#     Works with pandas Series/DataFrames.
#     """
#     var_P = sigma_P**2
#     a, b = beta_params_from_mean_var(p0, var_P)

#     # Determine the flip probability based on p0 relative to alpha
#     # Use np.where for vectorized conditional logic
#     # pflip_non_significant = 1.0 - beta.cdf(alpha, a, b) # if p0 < alpha, flip if P > alpha
#     pflip_significant = beta.cdf(alpha, a, b)  # if p0 >= alpha, flip if P <= alpha

#     # pflip = np.where(p0 < alpha, pflip_significant, pflip_non_significant)
#     pflip = pflip_significant

#     # Clip the result to ensure it's within [0, 1]
#     return np.clip(pflip, 0.0, 1.0)
def flip_proba_beta(p0, sigma_P, alpha=0.05):
    """
    Flip probability at alpha with P ~ Beta(a,b).
    Provide either sigma_P.
    Works with pandas Series/DataFrames.
    """
    p0 = np.asarray(p0, dtype=float)
    sigma_P = np.asarray(sigma_P, dtype=float)
    zero_variance = sigma_P <= 0
    var_P = sigma_P**2
    a, b = beta_params_from_mean_var(p0, var_P)

    # A significant observed p-value flips when its perturbed value exceeds
    # alpha; a non-significant value flips when the perturbed value is at most
    # alpha.  This is evaluated per lookup row using its own sigma_p.
    cdf_at_alpha = beta.cdf(alpha, a, b)
    pflip = np.where(p0 <= alpha, 1.0 - cdf_at_alpha, cdf_at_alpha)
    pflip = np.where(zero_variance, 0.0, pflip)

    # Clip the result to ensure it's within [0, 1]
    return np.clip(pflip, 0.0, 1.0)

In [14]:
p=flip_proba_beta(0.016639791628428845,0.0072,0.05)
p

np.float64(0.0005548877473058322)